# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qurban-12/ML_internship_Qurban/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the daily performance of one content item for one client on one report date. For this contract, I will use data from March 2026.

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", HF_TOKEN is not None)

HF token loaded: True


In [2]:
!pip install -q huggingface_hub


In [3]:
from huggingface_hub import login

login(token=HF_TOKEN)
print("Hugging Face login successful!")

Hugging Face login successful!


In [4]:
from huggingface_hub import list_repo_files

repo_id = "FlyRank/internship-warehouse"
repo_type = "dataset"

files = list_repo_files(repo_id=repo_id, repo_type=repo_type)

print("Number of files:", len(files))
print("\n".join(files[:20]))

Number of files: 24
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_da

In [5]:
for f in files:
    if "month=2026-03" in f:
        print(f)

fact_content_daily_performance/month=2026-03/data_0.parquet


In [6]:
from huggingface_hub import hf_hub_download
import pandas as pd

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)

df_march = pd.read_parquet(march_file)

print("Rows:", len(df_march))
print("Columns:", len(df_march.columns))
print("Date range:", df_march["report_date"].min(), "to", df_march["report_date"].max())

Rows: 9841378
Columns: 30
Date range: 2026-03-01 to 2026-03-31


In [7]:
print(df_march.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [8]:
grain_check = (
    df_march
    .groupby(["report_date", "client_hash_id", "content_hash_id"])
    .size()
    .reset_index(name="row_count")
)

duplicates = grain_check[grain_check["row_count"] > 1]

print("Duplicate grain combinations:", len(duplicates))

if len(duplicates) == 0:
    print("Grain check passed: one row = one report date + one client + one content item.")
else:
    print("Grain check failed. Duplicate combinations found:")
    display(duplicates.head())

Duplicate grain combinations: 0
Grain check passed: one row = one report date + one client + one content item.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [9]:
for i, col in enumerate(df_march.columns, start=1):
    print(i, col)

1 report_date
2 client_hash_id
3 content_hash_id
4 client_has_gsc
5 client_has_ga4
6 gsc_data_available
7 ga4_data_available
8 gsc_impressions
9 gsc_clicks
10 gsc_sum_position
11 gsc_avg_position
12 ga4_pageviews
13 ga4_sessions
14 ga4_users
15 ga4_engaged_sessions
16 ga4_total_engagement_sec
17 sessions_organic
18 sessions_direct
19 sessions_referral
20 sessions_social
21 sessions_paid
22 sessions_ai
23 ai_chatgpt
24 ai_perplexity
25 ai_gemini
26 ai_copilot
27 ai_claude
28 ai_meta
29 ai_other
30 scroll_events


Features:
- gsc_impressions — previous-day search impressions, available before the next-day prediction.
- gsc_avg_position — previous-day average Google Search position, available before the next-day prediction.
- ga4_sessions — previous-day sessions, available before the next-day prediction.
- ga4_engaged_sessions — previous-day engaged sessions, available before the next-day prediction.
- scroll_events — previous-day scroll events, available before the next-day prediction.

Label/proxy:
- next-day gsc_clicks — the Google Search clicks we want to predict for the following day.

Context:
- report_date — identifies the observation date.
- client_hash_id — identifies the pseudonymized client.
- content_hash_id — identifies the pseudonymized content item.
- client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available — indicate client and data availability.

Excluded:
- Individual AI-source fields (ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other) are excluded because this contract focuses on predicting Google Search clicks rather than individual AI-source traffic.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The March 2026 data has one row per report date, client, and content item. I verified this by checking for duplicate combinations of report_date, client_hash_id, and content_hash_id.


March 2026 contains 9,841,378 rows, covering 2026-03-01 through 2026-03-31.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain_check = (
    df_march
    .groupby(["report_date", "client_hash_id", "content_hash_id"])
    .size()
    .reset_index(name="row_count")
)

duplicates = grain_check[grain_check["row_count"] > 1]

print("Duplicate grain combinations:", len(duplicates))

if len(duplicates) == 0:
    print("Grain verified: one row = one report date + one client + one content item.")
else:
    display(duplicates.head())

print("Row count:", len(df_march))
print("Minimum date:", df_march["report_date"].min())
print("Maximum date:", df_march["report_date"].max())

Duplicate grain combinations: 0
Grain verified: one row = one report date + one client + one content item.
Row count: 9841378
Minimum date: 2026-03-01
Maximum date: 2026-03-31


After filtering March 2026 to rows where GSC data is available, I measured how many rows remain. The availability flag is checked explicitly with IS TRUE.

In [12]:
available_gsc = df_march[df_march["gsc_data_available"] == True]

print("Total March rows:", len(df_march))
print("Rows with GSC available (IS TRUE):", len(available_gsc))

Total March rows: 9841378
Rows with GSC available (IS TRUE): 3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

I will use five previous-day features to predict next-day Google Search clicks. The features are gsc_impressions, gsc_avg_position, ga4_sessions, ga4_engaged_sessions, and scroll_events. Each feature is shifted by one day within the same client and content item, so it is available before the next-day prediction. Rows are kept only when the previous day and next day are consecutive calendar days.


In [13]:
import pandas as pd

# Sort observations by client, content item, and date
work = df_march.sort_values(
    ["client_hash_id", "content_hash_id", "report_date"]
).copy()

group_cols = ["client_hash_id", "content_hash_id"]

# Check that the previous and next observations are actually
# the previous and next calendar days.
work["prev_date"] = work.groupby(group_cols)["report_date"].shift(1)
work["next_date"] = work.groupby(group_cols)["report_date"].shift(-1)

work["prev_day_ok"] = (
    work["prev_date"] == work["report_date"] - pd.Timedelta(days=1)
)

work["next_day_ok"] = (
    work["next_date"] == work["report_date"] + pd.Timedelta(days=1)
)

# Five features: use the previous day's value
feature_cols = [
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "scroll_events"
]

for col in feature_cols:
    work[f"{col}_prev_day"] = (
        work.groupby(group_cols)[col].shift(1)
    )

# Label: next day's Google Search clicks
work["gsc_clicks_next_day"] = (
    work.groupby(group_cols)["gsc_clicks"].shift(-1)
)

# Also shift availability flags so we know the feature data
# and target data were actually available.
work["gsc_available_prev_day"] = (
    work.groupby(group_cols)["gsc_data_available"].shift(1)
)

work["ga4_available_prev_day"] = (
    work.groupby(group_cols)["ga4_data_available"].shift(1)
)

work["gsc_available_next_day"] = (
    work.groupby(group_cols)["gsc_data_available"].shift(-1)
)

# Keep only observations with consecutive calendar days
# and the required data availability.
feature_frame = work[
    work["prev_day_ok"]
    & work["next_day_ok"]
    & (work["gsc_available_prev_day"] == True)
    & (work["ga4_available_prev_day"] == True)
    & (work["gsc_available_next_day"] == True)
].copy()

# Keep only the contract fields
feature_frame = feature_frame[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions_prev_day",
        "gsc_avg_position_prev_day",
        "ga4_sessions_prev_day",
        "ga4_engaged_sessions_prev_day",
        "scroll_events_prev_day",
        "gsc_clicks_next_day"
    ]
].reset_index(drop=True)

print("Feature-frame rows:", len(feature_frame))
print("Feature-frame columns:", len(feature_frame.columns))
display(feature_frame.head())

Feature-frame rows: 327758
Feature-frame columns: 9


,report_date,client_hash_id,content_hash_id,gsc_impressions_prev_day,gsc_avg_position_prev_day,ga4_sessions_prev_day,ga4_engaged_sessions_prev_day,scroll_events_prev_day,gsc_clicks_next_day
0,2026-03-25,client_08d2847f24cf89c1,content_0071174ee1f89d8a,13.0,11.384615,1.0,0.0,0.0,0.0
1,2026-03-03,client_08d2847f24cf89c1,content_07f1fdb8d2a0cf9d,38.0,6.000000,1.0,0.0,0.0,0.0
2,2026-03-04,client_08d2847f24cf89c1,content_07f1fdb8d2a0cf9d,94.0,8.446809,1.0,0.0,0.0,0.0
3,2026-03-08,client_08d2847f24cf89c1,content_07f1fdb8d2a0cf9d,11.0,14.727273,1.0,0.0,0.0,0.0
4,2026-03-13,client_08d2847f24cf89c1,content_07f1fdb8d2a0cf9d,3.0,9.333333,1.0,0.0,0.0,0.0


In [14]:
print("Missing values in feature frame:")

missing_values = feature_frame.isna().sum()

display(missing_values)

Missing values in feature frame:


,0
report_date,0
client_hash_id,0
content_hash_id,0
gsc_impressions_prev_day,0
gsc_avg_position_prev_day,0
ga4_sessions_prev_day,0
ga4_engaged_sessions_prev_day,0
scroll_events_prev_day,0
gsc_clicks_next_day,0


I will first measure a quick baseline using only the five contracted features. Then I will intentionally add the next-day click label itself as a feature to demonstrate label leakage. The leaked result should become artificially close to perfect. After observing this effect, the label-derived column will be removed and the honest five-feature result will be kept.


In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Work in date order so the test set represents later observations.
model_df = feature_frame.sort_values("report_date").copy()

feature_names = [
    "gsc_impressions_prev_day",
    "gsc_avg_position_prev_day",
    "ga4_sessions_prev_day",
    "ga4_engaged_sessions_prev_day",
    "scroll_events_prev_day"
]

target_name = "gsc_clicks_next_day"

# Use the first 80% for training and the last 20% for testing.
split_index = int(len(model_df) * 0.80)

train = model_df.iloc[:split_index]
test = model_df.iloc[split_index:]

X_train = train[feature_names]
X_test = test[feature_names]

y_train = train[target_name]
y_test = test[target_name]

# Honest five-feature model
honest_model = LinearRegression()
honest_model.fit(X_train, y_train)

honest_predictions = honest_model.predict(X_test)
honest_r2 = r2_score(y_test, honest_predictions)

print("Honest five-feature R²:", round(honest_r2, 4))

# Intentional leakage:
# Add the target itself as a feature.
X_train_leaky = X_train.copy()
X_test_leaky = X_test.copy()

X_train_leaky["LEAKED_NEXT_DAY_CLICKS"] = y_train.values
X_test_leaky["LEAKED_NEXT_DAY_CLICKS"] = y_test.values

leaky_model = LinearRegression()
leaky_model.fit(X_train_leaky, y_train)

leaky_predictions = leaky_model.predict(X_test_leaky)
leaky_r2 = r2_score(y_test, leaky_predictions)

print("Leaky model R²:", round(leaky_r2, 4))
print("R² jump from leakage:", round(leaky_r2 - honest_r2, 4))

Honest five-feature R²: 0.3276
Leaky model R²: 1.0
R² jump from leakage: 0.6724


The intentional leakage experiment increased R² from 0.3276 to 1.0000 because the next-day label was directly included as an input feature. This confirms that the perfect score was caused by leakage, not genuine predictive information. I therefore remove the label-derived column and keep only the five contracted previous-day features for the final frame.


In [16]:
# Keep only the honest five features and the target.
honest_feature_frame = feature_frame[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions_prev_day",
        "gsc_avg_position_prev_day",
        "ga4_sessions_prev_day",
        "ga4_engaged_sessions_prev_day",
        "scroll_events_prev_day",
        "gsc_clicks_next_day"
    ]
].copy()

print("Final honest feature frame:")
print("Rows:", len(honest_feature_frame))
print("Columns:", len(honest_feature_frame.columns))

display(honest_feature_frame.head())

Final honest feature frame:
Rows: 327758
Columns: 9


,report_date,client_hash_id,content_hash_id,gsc_impressions_prev_day,gsc_avg_position_prev_day,ga4_sessions_prev_day,ga4_engaged_sessions_prev_day,scroll_events_prev_day,gsc_clicks_next_day
0,2026-03-25,client_08d2847f24cf89c1,content_0071174ee1f89d8a,13.0,11.384615,1.0,0.0,0.0,0.0
1,2026-03-03,client_08d2847f24cf89c1,content_07f1fdb8d2a0cf9d,38.0,6.000000,1.0,0.0,0.0,0.0
2,2026-03-04,client_08d2847f24cf89c1,content_07f1fdb8d2a0cf9d,94.0,8.446809,1.0,0.0,0.0,0.0
3,2026-03-08,client_08d2847f24cf89c1,content_07f1fdb8d2a0cf9d,11.0,14.727273,1.0,0.0,0.0,0.0
4,2026-03-13,client_08d2847f24cf89c1,content_07f1fdb8d2a0cf9d,3.0,9.333333,1.0,0.0,0.0,0.0


All five features are previous-day measurements, so they are available before making the next-day prediction. GSC features are used only when GSC data was available on the previous day. GA4 features are used only when GA4 data was available on the previous day. The target is the following day's GSC clicks and is never used as an input feature.


In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

A key limitation is uneven data availability across clients and dates. The March 2026 data contains many rows without usable GSC or GA4 data, so the final feature frame represents only observations where the required previous-day data and next-day GSC data were available. Therefore, the results should not be interpreted as representing every client or every date equally. Missing availability should not be treated as zero traffic or zero activity.


The final contract uses one row per client-content prediction date, with five previous-day features to predict next-day Google Search clicks. The March 2026 verification confirmed the daily client-content grain, the date window, and GSC availability. The final frame contains 327,758 complete observations. The leakage experiment showed that including the target as a feature artificially increased R² from 0.3276 to 1.0000, so the leaked field was removed and the honest five-feature frame was retained.
